# Last-Mile Delivery Delay Prediction: Consolidated Notebook

This notebook contains the complete pipeline (preprocessing, weather-data merging, model tuning, stacking ensemble training, evaluation, diagnostics, feature importance, and SHAP-based feature explainability) in a single notebook.

### 🚀 Running on Google Colab / Local Jupyter Notebook:
1. **Colab File Upload**: Click the folder icon on the left panel in Google Colab, and upload all your raw CSV datasets (`train.csv`, `bengaluru.csv`, `bombay.csv`, etc.) directly into the root content directory.
2. **Local Notebook**: Put your CSV datasets in a folder named `data/raw/` in the same directory as this notebook.
3. Run the cells step-by-step!

### Section 1: Install & Import Dependencies

In [ ]:
# Install dependencies if running on Google Colab
!pip install -q geopy catboost shap

import os
import warnings
warnings.filterwarnings("ignore")
import glob
import joblib
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, RandomizedSearchCV, learning_curve
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor, StackingRegressor
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from catboost import CatBoostRegressor
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

print("All packages imported successfully!")

### Section 2: Configure Paths and Find Datasets

In [ ]:
# Detect environment
IS_COLAB = 'COLAB_GPU' in os.environ or os.path.exists('/content')

if IS_COLAB:
    # Colab expects uploaded datasets to be in Content root directory
    RAW_DIR = '/content'
    PROCESSED_DIR = '/content/processed'
    MODELS_DIR = '/content/models'
    print("Running on Google Colab!")
else:
    # Local notebook path defaults
    BASE_DIR = os.getcwd()
    RAW_DIR = os.path.join(BASE_DIR, 'data', 'raw')
    PROCESSED_DIR = os.path.join(BASE_DIR, 'data', 'processed')
    MODELS_DIR = os.path.join(BASE_DIR, 'models')
    print(f"Running locally in: {BASE_DIR}")

os.makedirs(PROCESSED_DIR, exist_ok=True)
os.makedirs(MODELS_DIR, exist_ok=True)

# Check for files
csv_files = glob.glob(os.path.join(RAW_DIR, "*.csv"))
print(f"Found {len(csv_files)} CSV files in {RAW_DIR}:")
for f in csv_files:
    print(f"  - {os.path.basename(f)}")

### Section 3: Raw Dataset Exploration (Data Inspection)

In [ ]:
# Find delivery dataset
delivery_paths = glob.glob(os.path.join(RAW_DIR, "*delivery*.csv")) + glob.glob(os.path.join(RAW_DIR, "*train*.csv"))
delivery_paths = [p for p in delivery_paths if 'weather' not in os.path.basename(p).lower()]

if not delivery_paths:
    print("[X] Error: Could not find delivery train.csv file! Please upload your dataset.")
else:
    print(f"Loading delivery raw dataset from: {delivery_paths[0]}...")
    df_delivery_raw = pd.read_csv(delivery_paths[0])

In [ ]:
# Inspect Delivery Dataset Schema (info)
if 'df_delivery_raw' in locals():
    print("\n=== Delivery Dataset Info ===")
    df_delivery_raw.info()

In [ ]:
# Inspect Delivery Dataset Numerical Distribution (describe)
if 'df_delivery_raw' in locals():
    from IPython.display import display
    print("\n=== Delivery Dataset Numerical Summary ===")
    display(df_delivery_raw.describe())

In [ ]:
# Inspect Delivery Dataset Sample Rows (head)
if 'df_delivery_raw' in locals():
    from IPython.display import display
    print("\n=== Delivery Dataset First 5 Rows ===")
    display(df_delivery_raw.head(5))

In [ ]:
# Find weather datasets
weather_files = glob.glob(os.path.join(RAW_DIR, "*.csv"))
weather_files = [f for f in weather_files if 'delivery' not in os.path.basename(f).lower() 
                 and 'train' not in os.path.basename(f).lower() 
                 and 'test' not in os.path.basename(f).lower()]

if weather_files:
    print(f"\nLoading sample weather raw dataset from: {weather_files[0]}...")
    df_weather_raw = pd.read_csv(weather_files[0])
else:
    print("\n[!] No weather CSV files found yet for data inspection.")

In [ ]:
# Inspect Weather Dataset Schema (info)
if 'df_weather_raw' in locals():
    print("\n=== Weather Dataset Info ===")
    df_weather_raw.info()

In [ ]:
# Inspect Weather Dataset Numerical Distribution (describe)
if 'df_weather_raw' in locals():
    from IPython.display import display
    print("\n=== Weather Dataset Numerical Summary ===")
    display(df_weather_raw.describe())

In [ ]:
# Inspect Weather Dataset Sample Rows (head)
if 'df_weather_raw' in locals():
    from IPython.display import display
    print("\n=== Weather Dataset First 5 Rows ===")
    display(df_weather_raw.head(5))

### Section 4: Data Preprocessing and Weather Merging Pipeline

In [ ]:
CITY_MAPPING = {
    'BANG': 'bengaluru', 'BEN': 'bengaluru', 'MUM': 'bombay', 'BOM': 'bombay',
    'DEL': 'delhi', 'HYD': 'hyderabad', 'JAIPUR': 'jaipur', 'JAI': 'jaipur',
    'KNP': 'kanpur', 'KAN': 'kanpur', 'NAG': 'nagpur', 'CHEN': 'chennai',
    'CHE': 'chennai', 'KOL': 'kolkata', 'COIMB': 'coimbatore', 'PUNE': 'pune',
    'PUN': 'pune', 'INDO': 'indore', 'IND': 'indore', 'SUR': 'surat',
    'LUDH': 'ludhiana', 'RANCHI': 'ranchi', 'AGRA': 'agra', 'ALH': 'allahabad',
    'AURG': 'aurangabad', 'BHO': 'bhopal', 'BHOP': 'bhopal', 'GOA': 'goa', 'VAD': 'vadodara'
}

def haversine_distance(lat1, lon1, lat2, lon2):
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = np.sin(dlat/2)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon/2)**2
    c = 2 * np.arcsin(np.sqrt(a))
    return c * 6371.0

def process_weather():
    weather_files = glob.glob(os.path.join(RAW_DIR, "*.csv"))
    weather_files = [f for f in weather_files if 'delivery' not in os.path.basename(f).lower() 
                     and 'train' not in os.path.basename(f).lower() 
                     and 'test' not in os.path.basename(f).lower()]
    if not weather_files:
        return None, None
    all_dfs = []
    for f in weather_files:
        city_name = os.splitext(os.path.basename(f))[0].lower()
        df = pd.read_csv(f)
        if 'date_time' in df.columns:
            df['date_time'] = pd.to_datetime(df['date_time'], errors='coerce')
            df = df.dropna(subset=['date_time'])
            df['city'] = city_name
            df['month'] = df['date_time'].dt.month
            df['hour'] = df['date_time'].dt.hour
            weather_cols = ['tempC', 'humidity', 'precipMM', 'windspeedKmph', 'pressure', 'cloudcover']
            existing = [c for c in weather_cols if c in df.columns]
            for col in existing:
                df[col] = pd.to_numeric(df[col], errors='coerce')
            all_dfs.append(df.groupby(['city', 'month', 'hour'])[existing].mean().reset_index())
    
    if not all_dfs:
        return None, None
    full_df = pd.concat(all_dfs, ignore_index=True)
    all_features = [c for c in full_df.columns if c not in ['city', 'month', 'hour']]
    national_fallback = full_df.groupby(['month', 'hour'])[all_features].mean().reset_index()
    return full_df, national_fallback

# Preprocess delivery data
print("Starting Preprocessing...")
delivery_train_paths = glob.glob(os.path.join(RAW_DIR, "*delivery*.csv")) + glob.glob(os.path.join(RAW_DIR, "*train*.csv"))
delivery_train_paths = [p for p in delivery_train_paths if 'weather' not in os.path.basename(p).lower()]

if not delivery_train_paths:
    print("Error: Could not find delivery train.csv file! Please upload your dataset.")
else:
    df_delivery = pd.read_csv(delivery_train_paths[0])
    for col in df_delivery.columns:
        if df_delivery[col].dtype == 'object':
            df_delivery[col] = df_delivery[col].astype(str).str.strip().replace(['NaN', 'nan'], np.nan)
            
    # Clean target
    target_col = 'Time_taken(min)'
    df_delivery[target_col] = df_delivery[target_col].astype(str).str.replace('(min)', '', regex=False).str.strip()
    df_delivery[target_col] = pd.to_numeric(df_delivery[target_col], errors='coerce')
    df_delivery = df_delivery.dropna(subset=[target_col])
    
    # Fill numeric columns
    df_delivery['Delivery_person_Age'] = pd.to_numeric(df_delivery['Delivery_person_Age'], errors='coerce')
    df_delivery['Delivery_person_Age'] = df_delivery['Delivery_person_Age'].fillna(df_delivery['Delivery_person_Age'].median())
    df_delivery['Delivery_person_Ratings'] = pd.to_numeric(df_delivery['Delivery_person_Ratings'], errors='coerce')
    df_delivery['Delivery_person_Ratings'] = df_delivery['Delivery_person_Ratings'].fillna(df_delivery['Delivery_person_Ratings'].mean())
    df_delivery['multiple_deliveries'] = pd.to_numeric(df_delivery['multiple_deliveries'], errors='coerce').fillna(0.0)
    
    # Clean coordinates
    coord_cols = ['Restaurant_latitude', 'Restaurant_longitude', 'Delivery_location_latitude', 'Delivery_location_longitude']
    for col in coord_cols:
        df_delivery[col] = pd.to_numeric(df_delivery[col], errors='coerce').abs()
    df_delivery = df_delivery.dropna(subset=coord_cols)
    
    # Haversine distance
    df_delivery['distance_km'] = haversine_distance(
        df_delivery['Restaurant_latitude'], df_delivery['Restaurant_longitude'],
        df_delivery['Delivery_location_latitude'], df_delivery['Delivery_location_longitude']
    )
    
    # City extraction
    df_delivery['city_code'] = df_delivery['Delivery_person_ID'].apply(lambda x: x.split('RES')[0].strip() if isinstance(x, str) and 'RES' in x else None)
    df_delivery['mapped_city_name'] = df_delivery['city_code'].map(CITY_MAPPING).fillna('unknown')
    
    # Temporal features
    df_delivery['parsed_date'] = pd.to_datetime(df_delivery['Order_Date'], format='%d-%m-%Y', errors='coerce')
    mask = df_delivery['parsed_date'].isna()
    if mask.any():
        df_delivery.loc[mask, 'parsed_date'] = pd.to_datetime(df_delivery.loc[mask, 'Order_Date'], errors='coerce')
    df_delivery = df_delivery.dropna(subset=['parsed_date'])
    df_delivery['order_month'] = df_delivery['parsed_date'].dt.month
    df_delivery['order_day_of_week'] = df_delivery['parsed_date'].dt.dayofweek
    df_delivery['order_day_of_month'] = df_delivery['parsed_date'].dt.day
    df_delivery['is_weekend'] = df_delivery['order_day_of_week'].isin([5, 6]).astype(int)
    
    def extract_h(x):
        if pd.isna(x) or not isinstance(x, str): return 12.0
        try:
            return float(x.split(':')[0]) if ':' in x else float(x.split('.')[0])
        except:
            return 12.0
    df_delivery['order_hour'] = df_delivery['Time_Orderd'].apply(extract_h)
    
    # Weather merge
    weather_df, national_profile = process_weather()
    if weather_df is not None:
        df_merged = pd.merge(df_delivery, weather_df, left_on=['mapped_city_name', 'order_month', 'order_hour'], right_on=['city', 'month', 'hour'], how='left')
        unmatched = df_merged['tempC'].isna()
        if unmatched.any() and national_profile is not None:
            w_cols = [c for c in weather_df.columns if c not in ['city', 'month', 'hour']]
            unmatched_df = df_merged[unmatched].drop(columns=w_cols + ['city', 'month', 'hour'], errors='ignore')
            matched_fallbacks = pd.merge(unmatched_df, national_profile, left_on=['order_month', 'order_hour'], right_on=['month', 'hour'], how='left')
            matched_cities = df_merged[~unmatched]
            for col in matched_cities.columns:
                if col not in matched_fallbacks.columns:
                    matched_fallbacks[col] = np.nan
            matched_fallbacks = matched_fallbacks[matched_cities.columns]
            df_merged = pd.concat([matched_cities, matched_fallbacks], ignore_index=True)
        df_merged = df_merged.drop(columns=['city', 'month', 'hour'], errors='ignore')
    else:
        df_merged = df_delivery.copy()
        for col in ['tempC', 'humidity', 'precipMM', 'windspeedKmph', 'pressure', 'cloudcover']:
            df_merged[col] = 0.0
            
    weather_features = ['tempC', 'humidity', 'precipMM', 'windspeedKmph', 'pressure', 'cloudcover']
    for col in weather_features:
        df_merged[col] = df_merged[col].fillna(df_merged[col].median() if not pd.isna(df_merged[col].median()) else 0.0)
        
    if 'Weatherconditions' in df_merged.columns:
        df_merged['Weatherconditions'] = df_merged['Weatherconditions'].astype(str).str.replace(r'conditions\\s*', '', regex=True).replace('nan', np.nan)
        df_merged['Weatherconditions'] = df_merged['Weatherconditions'].fillna(df_merged['Weatherconditions'].mode()[0])
        
    categorical_cols = ['Road_traffic_density', 'Type_of_order', 'Type_of_vehicle', 'Festival', 'City', 'Weatherconditions']
    for col in categorical_cols:
        if col in df_merged.columns:
            df_merged[col] = df_merged[col].fillna(df_merged[col].mode()[0] if not df_merged[col].mode().empty else 'unknown').astype(str)
            
    df_merged.to_csv(os.path.join(PROCESSED_DIR, "merged_delivery_weather.csv"), index=False)
    print(f"Preprocessing finished successfully! Processed data rows: {len(df_merged)}")

### Section 5: Train and Evaluate Base Models

In [ ]:
df_train = pd.read_csv(os.path.join(PROCESSED_DIR, "merged_delivery_weather.csv"))

numeric_features = ['Delivery_person_Age', 'Delivery_person_Ratings', 'multiple_deliveries', 'distance_km', 'order_month', 'order_day_of_week', 'order_day_of_month', 'is_weekend', 'order_hour', 'tempC', 'humidity', 'precipMM', 'windspeedKmph', 'pressure', 'cloudcover']
categorical_features = ['Weatherconditions', 'Road_traffic_density', 'Vehicle_condition', 'Type_of_order', 'Type_of_vehicle', 'Festival', 'City']
all_features = numeric_features + categorical_features

X = df_train[all_features]
y = df_train['Time_taken(min)']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Column Preprocessor
preprocessor = ColumnTransformer(transformers=[
    ('num', Pipeline([('imputer', SimpleImputer(strategy='median')), ('scaler', StandardScaler())]), numeric_features),
    ('cat', Pipeline([('imputer', SimpleImputer(strategy='most_frequent')), ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))]), categorical_features)
])

# Base models
base_models = {
    'RandomForest': Pipeline([('preprocessor', preprocessor), ('regressor', RandomForestRegressor(random_state=42, n_jobs=-1, max_depth=15))]),
    'LightGBM': Pipeline([('preprocessor', preprocessor), ('regressor', LGBMRegressor(random_state=42, n_jobs=-1, verbose=-1))]),
    'XGBoost': Pipeline([('preprocessor', preprocessor), ('regressor', XGBRegressor(random_state=42, n_jobs=-1))]),
    'CatBoost': Pipeline([('preprocessor', preprocessor), ('regressor', CatBoostRegressor(random_state=42, verbose=0, thread_count=-1, allow_writing_files=False))]),
    'Ridge': Pipeline([('preprocessor', preprocessor), ('regressor', Ridge())])
}

param_grids = {
    'RandomForest': {'regressor__n_estimators': [50, 100], 'regressor__min_samples_split': [2, 10]},
    'LightGBM': {'regressor__n_estimators': [100, 150], 'regressor__learning_rate': [0.05, 0.1]},
    'XGBoost': {'regressor__n_estimators': [100, 150], 'regressor__learning_rate': [0.05, 0.1]},
    'CatBoost': {'regressor__iterations': [100, 150], 'regressor__learning_rate': [0.05, 0.1]},
    'Ridge': {'regressor__alpha': [1.0, 10.0]}
}

tuned_models = {}
results = []

# Hyperparameter search tuning
for name, model_pipeline in base_models.items():
    print(f"Tuning {name}...")
    search = RandomizedSearchCV(model_pipeline, param_grids[name], n_iter=3, cv=3, scoring='neg_mean_squared_error', random_state=42, n_jobs=-1)
    search.fit(X_train, y_train)
    best_model = search.best_estimator_
    tuned_models[name] = best_model
    
    y_pred = best_model.predict(X_test)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    mae = mean_absolute_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)
    results.append({'Model': name, 'RMSE': rmse, 'MAE': mae, 'R2': r2})
    print(f"  Best params: {search.best_params_}")
    print(f"  Performance: RMSE={rmse:.3f}, MAE={mae:.3f}, R2={r2:.3f}")

### Section 6: Train and Save Stacking Ensemble

In [ ]:
# Stacking Ensemble
print("Training Stacking Ensemble...")
estimators = [(name, model.named_steps['regressor']) for name, model in tuned_models.items() if name != 'Ridge']
stack_reg = StackingRegressor(estimators=estimators, final_estimator=Ridge(alpha=1.0), n_jobs=-1, cv=3)
ensemble_pipeline = Pipeline([('preprocessor', preprocessor), ('regressor', stack_reg)])
ensemble_pipeline.fit(X_train, y_train)

y_pred_ens = ensemble_pipeline.predict(X_test)
rmse_ens = np.sqrt(mean_squared_error(y_test, y_pred_ens))
mae_ens = mean_absolute_error(y_test, y_pred_ens)
r2_ens = r2_score(y_test, y_pred_ens)
results.append({'Model': 'StackingEnsemble', 'RMSE': rmse_ens, 'MAE': mae_ens, 'R2': r2_ens})

print(f"Stacking Ensemble: RMSE={rmse_ens:.3f}, MAE={mae_ens:.3f}, R2={r2_ens:.3f}")

# Print summary comparison table
df_results = pd.DataFrame(results)
print("\n=== MODEL COMPARISON SUMMARY ===")
print(df_results.to_string(index=False))

# Save ensemble model
joblib.dump(ensemble_pipeline, os.path.join(MODELS_DIR, "ensemble_model.joblib"))
print("Model saved to models/ensemble_model.joblib")

### Section 7: Model Diagnostic Plots

In [ ]:
y_pred_ensemble = y_pred_ens

# A. Actual vs Predicted Scatter Plot
plt.figure(figsize=(8, 6))
plt.scatter(y_test, y_pred_ensemble, alpha=0.3, color='#34495e', edgecolors='none')
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], color='#e74c3c', linestyle='--', linewidth=2)
plt.title('Actual vs. Predicted Delivery Time (Stacking Ensemble)', fontsize=14)
plt.xlabel('Actual Time (min)', fontsize=12)
plt.ylabel('Predicted Time (min)', fontsize=12)
plt.grid(True, linestyle=':', alpha=0.6)
plt.tight_layout()
plt.show()

# B. Residuals Distribution Histogram
residuals = y_test - y_pred_ensemble
plt.figure(figsize=(8, 6))
sns.histplot(residuals, kde=True, color='#2ecc71', edgecolor='white', bins=30)
plt.axvline(0, color='#e74c3c', linestyle='--', linewidth=2)
plt.title('Residuals Distribution (Error Term Analysis)', fontsize=14)
plt.xlabel('Prediction Error (Actual - Predicted) in Minutes', fontsize=12)
plt.ylabel('Frequency', fontsize=12)
plt.grid(True, linestyle=':', alpha=0.6)
plt.tight_layout()
plt.show()

### Section 8: Learning Curve Analysis

In [ ]:
print("Calculating learning curves (using 10k samples subset for speed)...")
sample_size = min(10000, len(X))
X_sample = X.sample(n=sample_size, random_state=42)
y_sample = y.loc[X_sample.index]

train_sizes, train_scores, test_scores = learning_curve(
    tuned_models['LightGBM'], X_sample, y_sample, cv=3, n_jobs=-1,
    train_sizes=np.linspace(0.1, 1.0, 4), scoring='neg_mean_squared_error'
)
train_rmse = np.sqrt(-train_scores.mean(axis=1))
test_rmse = np.sqrt(-test_scores.mean(axis=1))

plt.figure(figsize=(8, 6))
plt.plot(train_sizes, train_rmse, 'o-', color="#e74c3c", linewidth=2, label="Training RMSE")
plt.plot(train_sizes, test_rmse, 's-', color="#2ecc71", linewidth=2, label="Cross-Validation RMSE")
plt.title("Learning Curves (LightGBM Estimator Convergence)", fontsize=14)
plt.xlabel("Training Set Size", fontsize=12)
plt.ylabel("RMSE (min)", fontsize=12)
plt.legend(loc="best", fontsize=11)
plt.grid(True, linestyle=':', alpha=0.6)
plt.tight_layout()
plt.show()

### Section 9: Visualize Feature Importance

In [ ]:
lgb_model = tuned_models['LightGBM']
feat_names = lgb_model.named_steps['preprocessor'].get_feature_names_out()
cleaned_names = [f.replace('num__', '').replace('cat__', '') for f in feat_names]
importances = lgb_model.named_steps['regressor'].feature_importances_

df_imp = pd.DataFrame({'Feature': cleaned_names, 'Importance': importances}).sort_values(by='Importance', ascending=False)

plt.figure(figsize=(10, 6))
sns.barplot(data=df_imp.head(15), x='Importance', y='Feature', palette='viridis')
plt.title('Top 15 Feature Importances (LightGBM)')
plt.xlabel('Importance Value')
plt.ylabel('Features')
plt.tight_layout()
plt.show()

### Section 10: Explainable AI (XAI) using SHAP

In [ ]:
import shap

# Extract preprocessor and fitted LightGBM from the ensemble pipeline
preprocessor = ensemble_pipeline.named_steps['preprocessor']
lgb_model = ensemble_pipeline.named_steps['regressor'].named_estimators_['LightGBM']

# Transform test data and clean feature names for readability
X_test_transformed = preprocessor.transform(X_test)
feature_names = [f.replace('num__', '').replace('cat__', '') for f in preprocessor.get_feature_names_out()]
X_test_transformed_df = pd.DataFrame(X_test_transformed, columns=feature_names)

# Initialize SHAP explainer and compute values
explainer = shap.TreeExplainer(lgb_model)
shap_values = explainer(X_test_transformed_df)

# Plot 1: SHAP summary graph (Global Interpretation)
plt.figure(figsize=(10, 6))
shap.summary_plot(shap_values, X_test_transformed_df, show=False)
plt.title("Global Feature Impact (SHAP Summary Plot)", fontsize=14)
plt.tight_layout()
plt.show()

# Plot 2: SHAP waterfall graph (Local Interpretation for the first sample in the test set)
plt.figure(figsize=(10, 6))
shap.plots.waterfall(shap_values[0], show=False)
plt.title("Decision Breakdown for a Single Test Prediction", fontsize=14)
plt.tight_layout()
plt.show()